# SynBioCrow 2.4.14 — Evidence-Aware Route Discrimination
Generation-free development ranking. Upload only the current 2.4.14 SOURCE ZIP. The notebook automatically locates the latest complete persisted development artifact directory in Google Drive. Validation/evaluation truth remains sealed.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

drive.mount("/content/drive", force_remount=False)
T0=time.time()
REPO=Path("/content/SynBioCrow_2_4_14")
SRCROOT=Path("/content/synbiocrow_2_4_14_source")
RUNOUT=Path("/content/synbiocrow_2_4_14_outputs")
DRIVE_BASE=Path("/content/drive/MyDrive/SynBioCrow/2.4")
DRIVE_OUT=DRIVE_BASE/"discrimination_2_4_14"
DRIVE_OUT.mkdir(parents=True,exist_ok=True)

def status(p,m):
    print(f"[2.4.14] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)

def run(cmd,cwd=None,timeout=1800):
    print("$"," ".join(map(str,cmd)),flush=True)
    env=os.environ.copy()
    if cwd:
        env["PYTHONPATH"]=str(cwd)+os.pathsep+env.get("PYTHONPATH","")
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,env=env,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

status(3,"upload current 2.4.14 SOURCE ZIP only")
uploaded=files.upload()
if len(uploaded)!=1:
    raise RuntimeError("Upload exactly one SynBioCrow 2.4.14 SOURCE ZIP")
src_name=next(iter(uploaded))
if not src_name.lower().endswith(".zip"):
    raise RuntimeError("SOURCE artifact must be a ZIP")
src_zip=Path("/content")/src_name
src_zip.write_bytes(uploaded[src_name])
src_sha=hashlib.sha256(src_zip.read_bytes()).hexdigest()
print("SOURCE:",src_name,"sha256=",src_sha,flush=True)

status(10,"clone develop/2.4 base")
shutil.rmtree(REPO,ignore_errors=True)
run(["git","clone","--depth","1","--branch","develop/2.4",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

status(18,"overlay 2.4.14 SOURCE delta")
shutil.rmtree(SRCROOT,ignore_errors=True)
SRCROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(src_zip) as z:
    z.extractall(SRCROOT)
for p in SRCROOT.rglob("*"):
    if p.is_file() and p.name not in {"README_SOURCE.txt"}:
        rel=p.relative_to(SRCROOT)
        dst=REPO/rel
        dst.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,dst)

status(25,"install lightweight SynBioCrow core only")
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)

status(35,"bounded discovery of complete development artifacts")
manifest=json.loads((REPO/"benchmarks/2.4/split_manifest_2_4_11.json").read_text())
dev_ids=[r["record_id"] for r in manifest["records"] if r["split"]=="development"]
sealed=[r["record_id"] for r in manifest["records"] if r["split"]!="development"]
print("development ids:",dev_ids,flush=True)
print("sealed nondevelopment ids:",sealed,flush=True)

dirs=[p for p in DRIVE_BASE.glob("development_routes_*") if p.is_dir()]
dirs.sort(key=lambda p:p.stat().st_mtime,reverse=True)
chosen=None
for d in dirs[:20]:
    if all((d/f"{rid}.json").is_file() for rid in dev_ids):
        chosen=d
        break
if chosen is None:
    raise FileNotFoundError(
        "No complete development_routes_* directory found under "
        "/content/drive/MyDrive/SynBioCrow/2.4"
    )
print("development artifact directory:",chosen,flush=True)

# Firewall check: the runner will open only the four exact development files.
for rid in sealed:
    print("SEALED - not opened:",rid,flush=True)

status(50,"run evidence-aware route discrimination")
shutil.rmtree(RUNOUT,ignore_errors=True)
RUNOUT.mkdir(parents=True,exist_ok=True)
run([sys.executable,str(REPO/"scripts/v24_rank_development_routes_2_4_14.py"),
     "--input-dir",str(chosen),
     "--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),
     "--out-dir",str(RUNOUT)],cwd=REPO,timeout=1800)

status(72,"verify scientific firewall")
summary=json.loads((RUNOUT/"discrimination_summary.json").read_text())
policy=json.loads((RUNOUT/"ranking_policy.json").read_text())
assert summary["generation_invoked"] is False
assert summary["validation_truth_accessed"] is False
assert summary["evaluation_truth_accessed"] is False
assert summary["tuning_performed"] is False
assert policy["generation_invoked"] is False
assert policy["validation_truth_accessed"] is False
assert policy["evaluation_truth_accessed"] is False
assert set(summary["development_record_ids"])==set(dev_ids)
assert set(summary["sealed_nondevelopment_record_ids"])==set(sealed)
print("ranked routes:",summary["ranked_route_count"],flush=True)
print("by target:",summary["ranked_route_count_by_record"],flush=True)

status(85,"persist outputs to Drive")
for p in RUNOUT.iterdir():
    if p.is_file():
        shutil.copy2(p,DRIVE_OUT/p.name)
run_manifest={
    "version":"2.4.14",
    "source_zip":src_name,
    "source_sha256":src_sha,
    "input_directory":str(chosen),
    "generation_invoked":False,
    "validation_truth_accessed":False,
    "evaluation_truth_accessed":False,
    "tuning_performed":False,
}
(DRIVE_OUT/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2,sort_keys=True)+"\n")

bundle=DRIVE_OUT/"SynBioCrow_2_4_14_EVIDENCE_AWARE_ROUTE_DISCRIMINATION_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(DRIVE_OUT.iterdir()):
        if p.is_file() and p!=bundle:
            z.write(p,p.name)

status(100,"PASS")
print("bundle:",bundle)
print("sha256:",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))


## Rescue download


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/discrimination_2_4_14/SynBioCrow_2_4_14_EVIDENCE_AWARE_ROUTE_DISCRIMINATION_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
